# E3 — v2 revision note (target-leakage re-run)

This notebook is `04_padding_mask.ipynb` with **one** change, required because
the leakage fix moved NF-UNSW-NB15 onto its own data version.

The version tag was a single string (`DATA_VER`) applied to every corpus. It
now resolves **per corpus** from `data_version_overrides` in the run config,
by prefix match: NSL-KDD stays on `v3_groupsplit`, NF-UNSW-NB15 uses
`v4_nofeatleak`. Three places follow from that — the cache path (cell 4), the
checkpoint-coverage search, which is now computed per corpus (cell 4), and the
checkpoint paths in the validity check and the grid (cells 5 and 6), which go
through a new `ckpt_for()` helper.

Why it mattered: with one tag, no suffix covers both corpora. The old search
picked whichever covered more and reported the other corpus's four cells as
missing, which gates the LaTeX export off without a single numeric mismatch
to explain it. Worse, the cache lookup would have silently loaded the
**stale, leaky** NF-UNSW cache, and the notebook would have looked healthy
while re-deriving the very numbers the fix was meant to replace.

Nothing else is touched: the architectures, the attacks, the masking premise
assertion, the validity check and the aggregation are unchanged, so the
masked/unmasked comparison keeps the semantics the manuscript reports.

The validity check now compares against `master_results_20261021_031658.csv`,
the post-fix run, which is the newest frame on Drive and therefore selected
automatically.

---

# E3 — Padding-Masked Attack Re-evaluation

**Question.** The pseudo-sequence representation pads every record to
$L=64$. NSL-KDD contributes $d=41$ real features and NF-UNSW-NB15-v2
$d=42$, so $23$ and $22$ trailing coordinates ($36\%$ and $34\%$) are
identically zero. The $\ell_\infty$ ball extends over all $64$
coordinates, so part of every perturbation budget is spent attacking
positions that carry no information.

**How much of the measured vulnerability is that artefact?**

This notebook re-runs the white-box grid with the perturbation confined
to the $d$ real coordinates and compares it against the published
unmasked result. Nothing is retrained: the released checkpoints and the
cached preprocessed tensors are reused, and the attack hyperparameters
are read from the published run configuration.

CSE-CIC-IDS2018 has $d=79>L$ and is truncated rather than padded, so it
has no padding coordinates and is excluded by construction.

**Validity check.** Cell 5 reproduces the published unmasked numbers.
If it fails, the notebook stops: the masked arm is only interpretable
once the harness is shown to reproduce the original measurement.

Run order: top to bottom. Expect roughly one hour on an A100.


In [ ]:
# === 1. ENVIRONMENT, PATHS AND GUARDS ==========================
import os, sys, json, time, itertools, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

try:
    from google.colab import drive  # noqa
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except Exception:
    DRIVE = os.environ.get("BTD_DRIVE", ".")
    print("[INFO] Not on Colab; DRIVE =", DRIVE)

# Locate the project folder. The run that produced the manuscript lived in
# Mamba_BTD; earlier drafts used other names, so several are tried and the
# one that actually contains cache/ and checkpoints/ wins.
CANDIDATES = ["Mamba_BTD", "BTD_Mamba_NIDS_Robustness", "Mamba_NIDS",
              "BTD_Mamba", "MAMBA_BTD"]
BASE = None
for root in [DRIVE, os.path.join(DRIVE, "Colab Notebooks")]:
    for cand in CANDIDATES:
        p = os.path.join(root, cand)
        for sub in ["", "BTD_Mamba_NIDS_Robustness"]:
            q = os.path.join(p, sub) if sub else p
            if os.path.isdir(os.path.join(q, "checkpoints")) and \
               os.path.isdir(os.path.join(q, "cache")):
                BASE = q
                break
        if BASE: break
    if BASE: break

if BASE is None:
    # last resort: walk the Drive looking for the signature pair
    for dirpath, dirnames, _ in os.walk(DRIVE):
        if "checkpoints" in dirnames and "cache" in dirnames:
            BASE = dirpath
            break

assert BASE is not None, (
    "Project folder not found. Set BASE manually to the directory that "
    "contains cache/ and checkpoints/.")
print("BASE =", BASE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
assert DEVICE == "cuda", (
    "This notebook is written for a GPU. On CPU the selective scan makes "
    "the grid take hundreds of hours.")

# Attack grid, taken from the published run configuration rather than
# retyped, so a mismatch is impossible.
cfgs = sorted(f for f in os.listdir(os.path.join(BASE, "logs"))
              if f.startswith("config_")) if os.path.isdir(os.path.join(BASE, "logs")) else []
if cfgs:
    RUNCFG = json.load(open(os.path.join(BASE, "logs", cfgs[-1])))
    print("config:", cfgs[-1])
else:
    RUNCFG = {}
SEQ_LEN   = RUNCFG.get("seq_len", 64)
D_MODEL   = RUNCFG.get("d_model", 64)
N_LAYERS  = RUNCFG.get("n_layers", 3)
ATK       = RUNCFG.get("attacks", {"FGSM": {"eps_list": [0.01, 0.03, 0.05, 0.1]},
                                   "PGD": {"eps_list": [0.01, 0.03, 0.05, 0.1],
                                           "alpha": 0.01, "steps": 10}})
EPS_LIST  = ATK["FGSM"]["eps_list"]
PGD_ALPHA = ATK["PGD"].get("alpha", 0.01)
PGD_STEPS = ATK["PGD"].get("steps", 10)
DATA_VER  = RUNCFG.get("data_version", "v3_groupsplit")
# v5 revision -- THE VERSION TAG IS NOW PER CORPUS.  The target-leakage fix
# changed only NF-UNSW-NB15's preprocessing (the `Attack` column was reaching
# the feature matrix), so that corpus moved to 'v4_nofeatleak' while NSL-KDD
# stayed on 'v3_groupsplit'.  A single tag cannot address both: used alone it
# would load the STALE, LEAKY cache for one of them and still look healthy.
DATA_VER_OVERRIDES = dict(RUNCFG.get("data_version_overrides", {}))

def data_ver_for(name: str) -> str:
    """Per-corpus version tag, matched by prefix so that derived dataset
    names (e.g. '<name>_ablation_dm32') resolve to the same override."""
    if name in DATA_VER_OVERRIDES:
        return DATA_VER_OVERRIDES[name]
    for base, ver in DATA_VER_OVERRIDES.items():
        if name.startswith(base + "_"):
            return ver
    return DATA_VER

# ---- the knobs you may want to change -------------------------------
DATASETS = ["NSL-KDD", "NF-UNSW-NB15"]   # CICIDS2018 has no padding
MODELS   = ["Mamba", "Transformer", "CNN1D", "LSTM"]
SEEDS    = [13, 42]        # extend to the full seven if the result is borderline
BATCH    = 512
LIMIT    = 0               # 0 = full test set
# ---------------------------------------------------------------------

print("data_version:", DATA_VER, "| per-corpus overrides:", DATA_VER_OVERRIDES or "(none)")
print(f"L={SEQ_LEN} d_model={D_MODEL} n_layers={N_LAYERS} "
      f"eps={EPS_LIST} pgd=(alpha={PGD_ALPHA}, steps={PGD_STEPS})")
print("datasets:", DATASETS, "| models:", MODELS, "| seeds:", SEEDS)

# --- pandas pickle compatibility --------------------------------------------
# The .pt caches embed a little pandas metadata alongside the tensors.
# pandas >= 2.3 pickles StringDtype as StringDtype(storage, na_value);
# pandas <= 2.2 accepts only StringDtype(storage), so a cache written by the
# newer pandas fails to load on an older runtime with
#   TypeError: StringDtype.__init__() takes from 1 to 2 positional arguments
# We tolerate the surplus constructor arguments.  This touches ONLY the
# unused metadata objects -- X_train / X_test / y_test / n_features, the
# values this notebook actually reads, are returned bit-for-bit unchanged.
def _install_pandas_pickle_shim():
    try:
        import pandas as pd
    except Exception:
        print("pandas not importable; skipping pickle shim")
        return
    targets = []
    for mod, nm in [("pandas.core.arrays.string_", "StringDtype"),
                    ("pandas.core.dtypes.dtypes", "ArrowDtype")]:
        try:
            targets.append(getattr(__import__(mod, fromlist=[nm]), nm))
        except Exception:
            pass
    patched_names = []
    for cls in targets:
        orig = cls.__init__
        if getattr(orig, "_e3_shim", False):
            continue
        def _make(orig):
            def _patched(self, *a, **kw):
                try:
                    return orig(self, *a, **kw)
                except TypeError:
                    kw.pop("na_value", None)
                    for n in range(len(a) - 1, -1, -1):
                        try:
                            return orig(self, *a[:n], **kw)
                        except TypeError:
                            continue
                    raise
            _patched._e3_shim = True
            return _patched
        cls.__init__ = _make(orig)
        patched_names.append(cls.__name__)
    print(f"pandas {pd.__version__}: pickle shim installed for "
          f"{', '.join(patched_names) if patched_names else '(nothing needed)'}")

_install_pandas_pickle_shim()

import hashlib

# Match the numerical environment of the published run.  That pipeline set
# TF32 matrix multiplication and cuDNN autotuning globally.  On an A100 TF32
# retains about ten mantissa bits against float32's twenty-four, so running
# without it shifts logits enough to flip predictions near the decision
# boundary -- which is the most likely reason a faithful-looking harness
# fails to reproduce the published accuracies.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# generate_adversarial_examples() in the published run built perturbations in
# batches of 256.  For FGSM the batch size is numerically irrelevant, but PGD
# draws its random start per batch, so the batch geometry must match.
ATTACK_BS = 256
print("TF32 enabled | cudnn.benchmark on | ATTACK_BS =", ATTACK_BS)


In [ ]:
# === 2. ARCHITECTURES (verbatim from the released pipeline) =====
# Reproduced exactly so that state dicts load with strict=True; any
# divergence would surface immediately as a key mismatch.

class SelectiveSSMBlock(nn.Module):
    def __init__(self, d_model, d_state=16, expand=2):
        super().__init__()
        self.d_model, self.d_inner, self.d_state = d_model, expand * d_model, d_state
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=4,
                                padding=3, groups=self.d_inner)
        self.x_proj = nn.Linear(self.d_inner, self.d_state * 2 + self.d_inner)
        self.dt_proj = nn.Linear(self.d_inner, self.d_inner)
        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, L, _ = x.shape
        residual = x
        x_in, z = self.in_proj(x).chunk(2, dim=-1)
        x_conv = self.conv1d(x_in.transpose(1, 2))[:, :, :L]
        x_conv = F.silu(x_conv.transpose(1, 2))
        dbc = self.x_proj(x_conv)
        delta, Bm, Cm = torch.split(
            dbc, [self.d_inner, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta))
        A = -torch.exp(self.A_log)
        dA = torch.exp(delta.unsqueeze(-1) * A)
        dB = delta.unsqueeze(-1) * Bm.unsqueeze(2)
        h = torch.zeros(B, self.d_inner, self.d_state,
                        device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            h = dA[:, t] * h + dB[:, t] * x_conv[:, t].unsqueeze(-1)
            ys.append(torch.einsum("bds,bs->bd", h, Cm[:, t]))
        y = torch.stack(ys, dim=1)
        y = (y + x_conv * self.D) * F.silu(z)
        return self.norm(self.out_proj(y) + residual)


class MambaNIDS(nn.Module):
    def __init__(self, seq_len, d_model=64, n_layers=3, d_state=16, n_classes=2):
        super().__init__()
        self.embed = nn.Linear(1, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len, d_model))
        self.blocks = nn.ModuleList(
            [SelectiveSSMBlock(d_model, d_state) for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2),
            nn.GELU(), nn.Linear(d_model // 2, n_classes))

    def forward(self, x):
        h = self.embed(x) + self.pos_embed
        for blk in self.blocks:
            h = blk(h)
        return self.head(h.mean(dim=1))


class TransformerNIDS(nn.Module):
    def __init__(self, seq_len, d_model=64, n_layers=3, n_heads=4, n_classes=2):
        super().__init__()
        self.embed = nn.Linear(1, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len, d_model))
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2),
            nn.GELU(), nn.Linear(d_model // 2, n_classes))

    def forward(self, x):
        h = self.encoder(self.embed(x) + self.pos_embed)
        return self.head(h.mean(dim=1))


class CNN1DNIDS(nn.Module):
    def __init__(self, seq_len, d_model=64, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, d_model, kernel_size=5, padding=2),
            nn.BatchNorm1d(d_model), nn.ReLU(),
            nn.Conv1d(d_model, d_model, kernel_size=5, padding=2),
            nn.BatchNorm1d(d_model), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1))
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Linear(d_model // 2, n_classes))

    def forward(self, x):
        return self.head(self.net(x.transpose(1, 2)).squeeze(-1))


class LSTMNIDS(nn.Module):
    def __init__(self, seq_len, d_model=64, n_layers=2, n_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(1, d_model, num_layers=n_layers,
                            batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, d_model),
            nn.GELU(), nn.Linear(d_model, n_classes))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out.mean(dim=1))


def build_model(name):
    if name == "Mamba":
        return MambaNIDS(SEQ_LEN, D_MODEL, N_LAYERS)
    if name == "Transformer":
        return TransformerNIDS(SEQ_LEN, D_MODEL, N_LAYERS)
    if name == "CNN1D":
        return CNN1DNIDS(SEQ_LEN, D_MODEL)
    if name == "LSTM":
        return LSTMNIDS(SEQ_LEN, D_MODEL, n_layers=max(1, N_LAYERS - 1))
    raise ValueError(name)


class ModelWrapper(nn.Module):
    """Input clamping exactly as in the published attack pipeline."""
    def __init__(self, model, clip_min, clip_max):
        super().__init__()
        self.model, self.clip_min, self.clip_max = model, clip_min, clip_max

    def forward(self, x):
        return self.model(torch.clamp(x, self.clip_min, self.clip_max))


print("architectures defined")


In [ ]:
# === 3. ATTACKS, WITH AN OPTIONAL COORDINATE MASK ==============
# mask=None reproduces the published attack exactly. mask=m confines the
# perturbation to the coordinates where m==1, so the padding positions
# retain their clean value of zero at every step.

def fgsm(model, x, y, eps, lo, hi, mask=None):
    x = x.clone().detach().requires_grad_(True)
    with torch.backends.cudnn.flags(enabled=False):
        loss = F.cross_entropy(model(x), y)
        grad = torch.autograd.grad(loss, x)[0]
    step = eps * grad.sign()
    if mask is not None:
        step = step * mask
    return torch.clamp(x.detach() + step, lo, hi)


def _start_noise(shape, dtype, device, eps, gen):
    """PGD's random start, drawn on the CPU from an explicit generator so that
    the masked and unmasked arms of a cell receive the IDENTICAL start.  With
    an unseeded start the two arms differ by random-start noise as well as by
    the mask, and the delta E3 measures would be confounded."""
    n = torch.empty(shape, dtype=torch.float32)
    n.uniform_(-eps, eps, generator=gen)
    return n.to(device=device, dtype=dtype)


def pgd(model, x, y, eps, alpha, steps, lo, hi, mask=None, gen=None):
    x0 = x.clone().detach()
    noise = (_start_noise(x0.shape, x0.dtype, x0.device, eps, gen)
             if gen is not None else torch.empty_like(x0).uniform_(-eps, eps))
    if mask is not None:
        noise = noise * mask
    x_adv = torch.clamp(x0 + noise, lo, hi).detach()
    for _ in range(steps):
        x_adv.requires_grad_(True)
        with torch.backends.cudnn.flags(enabled=False):
            loss = F.cross_entropy(model(x_adv), y)
            grad = torch.autograd.grad(loss, x_adv)[0]
        with torch.no_grad():
            step = alpha * grad.sign()
            if mask is not None:
                step = step * mask
            x_adv = x_adv + step
            delta = torch.clamp(x_adv - x0, -eps, eps)
            if mask is not None:
                delta = delta * mask
            x_adv = torch.clamp(x0 + delta, lo, hi)
    return x_adv.detach()


@torch.no_grad()
def _acc(model, X, y, bs):
    c = 0
    for i in range(0, len(X), bs):
        c += (model(X[i:i + bs]).argmax(1) == y[i:i + bs]).sum().item()
    return c / len(X)


def _tag_seed(tag, i):
    """Stable across processes, unlike the salted builtin hash()."""
    h = hashlib.blake2b(f"{tag}|{i}".encode(), digest_size=8).digest()
    return int.from_bytes(h, "big") % (2 ** 31 - 1)


def attacked_acc(model, X, y, bs, kind, eps, lo, hi, mask, tag=None):
    c = 0
    for i in range(0, len(X), bs):
        xb, yb = X[i:i + bs], y[i:i + bs]
        if kind == "FGSM":
            xa = fgsm(model, xb, yb, eps, lo, hi, mask)
        else:
            g = (torch.Generator().manual_seed(_tag_seed(tag, i))
                 if tag is not None else None)
            xa = pgd(model, xb, yb, eps, PGD_ALPHA, PGD_STEPS, lo, hi, mask,
                     gen=g)
        with torch.no_grad():
            c += (model(xa).argmax(1) == yb).sum().item()
    return c / len(X)


print("attacks defined")


In [ ]:
# === 4. LOAD CACHED TENSORS AND CHECK THE PADDING PREMISE =======
# The premise of E3 is that the trailing L-d coordinates are identically
# zero. That is asserted here rather than assumed.

DATA = {}
for name in DATASETS:
    path = os.path.join(BASE, "cache", f"{name}_seq{SEQ_LEN}_{data_ver_for(name)}.pt")
    assert os.path.exists(path), f"cache missing: {path}"
    try:
        c = torch.load(path, weights_only=False, map_location="cpu")
    except TypeError as e:
        raise RuntimeError(
            f"cache {path} was pickled by a different pandas than this "
            f"runtime provides ({e}). Re-run cell 1 to install the shim, "
            f"or pin pandas>=2.3 and restart the runtime."
        ) from None
    d = int(c["n_features"])
    Xtr, Xte, yte = c["X_train"], c["X_test"], c["y_test"]
    lo, hi = float(Xtr.min()), float(Xtr.max())
    n_pad = SEQ_LEN - min(d, SEQ_LEN)
    assert n_pad > 0, f"{name}: d={d} >= L={SEQ_LEN}; no padding, exclude it"
    pad_block = Xte[:, d:, :]
    assert bool((pad_block == 0).all()), (
        f"{name}: trailing coordinates are not identically zero; "
        "the masking premise does not hold")
    mask = torch.zeros(1, SEQ_LEN, 1)
    mask[:, :d, :] = 1.0
    DATA[name] = dict(X=Xte, y=yte, d=d, lo=lo, hi=hi, n_pad=n_pad,
                      mask=mask, n=len(Xte))
    print(f"{name:<16} d={d:<3} padding={n_pad} ({100*n_pad/SEQ_LEN:4.1f}%) "
          f"test={len(Xte)}  clip=[{lo:.4f}, {hi:.4f}]  padding all zero OK")

# Published results, used for the validity check and for the final table.
res = sorted(f for f in os.listdir(os.path.join(BASE, "results"))
             if f.startswith("master_results_"))
MASTER = pd.read_csv(os.path.join(BASE, "results", res[-1]))
RUN_ID = res[-1].replace("master_results_", "").replace(".csv", "")
print("\npublished results:", res[-1], "| rows:", len(MASTER))

# --- inventory, and a self-correcting version tag ---------------------------
# The tags come from the newest config_*.json (one default plus per-corpus
# overrides).  If the checkpoints on disk carry a different suffix the grid
# would report "missing" without a single numeric mismatch, so the suffix
# that actually covers each corpus's expected files is adopted automatically
# rather than left for the reader to fix.
ck_dir = os.path.join(BASE, "checkpoints")
have = sorted(f for f in os.listdir(ck_dir) if f.endswith(".pt"))
WANT = [(n, m, s) for n in DATASETS for m in MODELS for s in SEEDS]

# v5 revision: coverage is resolved PER CORPUS.  With NSL-KDD on
# 'v3_groupsplit' and NF-UNSW-NB15 on 'v4_nofeatleak', no single suffix
# covers both, and the former single-tag search would have reported half the
# grid as "missing" without one numeric mismatch to explain it.
def _coverage(ver, ds=None):
    return sum(f"{n}_{m}_seed{s}_{ver}.pt" in have
               for n, m, s in WANT if ds is None or n == ds)

suffixes = sorted({f.split("_seed", 1)[1].split("_", 1)[1][:-3]
                   for f in have if "_seed" in f and "_" in f.split("_seed", 1)[1]})
print(f"\ncheckpoints on disk : {len(have)}")
print(f"suffixes present    : {suffixes}")

RESOLVED_VER = {}
for _ds in DATASETS:
    _n_ds = sum(1 for n, _, _ in WANT if n == _ds)
    _cfg = data_ver_for(_ds)
    _best = max(suffixes, key=lambda v: _coverage(v, _ds)) if suffixes else _cfg
    _use = _best if _coverage(_best, _ds) > _coverage(_cfg, _ds) else _cfg
    RESOLVED_VER[_ds] = _use
    _note = "" if _use == _cfg else f"  !! switched from config value '{_cfg}'"
    print(f"  {_ds:<16} -> {_use:<16} covers {_coverage(_use, _ds)}/{_n_ds}{_note}")


def ckpt_for(name: str, mname: str, seed: int) -> str:
    """Checkpoint path under that corpus's own resolved version tag."""
    return os.path.join(BASE, "checkpoints",
                        f"{name}_{mname}_seed{seed}_{RESOLVED_VER[name]}.pt")


missing = [os.path.basename(ckpt_for(n, m, s)) for n, m, s in WANT
           if not os.path.exists(ckpt_for(n, m, s))]
if missing:
    print(f"!! {len(missing)}/{len(WANT)} expected checkpoints still missing; "
          f"those cells will be skipped. First few:")
    for _p in missing[:6]:
        print("   ", _p)
else:
    print("all expected checkpoints present")


In [ ]:
# === 5. VALIDITY CHECK: REPRODUCE THE PUBLISHED UNMASKED ARM ====
# One (dataset, model, seed) cell per dataset is re-attacked without a
# mask and compared against master_results. A mismatch means the harness
# differs from the pipeline that produced the manuscript, in which case
# nothing below is interpretable.

TOL = 2e-3
val_rows, VALID, MISSING_CK = [], True, []
for name in DATASETS:
    D = DATA[name]
    for mname in MODELS:
        seed = SEEDS[0]
        ck = ckpt_for(name, mname, seed)
        if not os.path.exists(ck):
            print(f"  !! checkpoint missing: {os.path.basename(ck)}")
            MISSING_CK.append(os.path.basename(ck))
            VALID = False
            continue
        net = build_model(mname)
        obj = torch.load(ck, map_location="cpu", weights_only=False)
        sd = obj
        if isinstance(obj, dict):
            for k in ("model_state", "state_dict", "model_state_dict"):
                if k in obj:
                    sd = obj[k]
                    break
        net.load_state_dict(sd, strict=True)
        net = net.to(DEVICE).eval()
        model = ModelWrapper(net, D["lo"], D["hi"]).eval()
        X, y = D["X"].to(DEVICE), D["y"].long().to(DEVICE)

        # The published pipeline measured clean accuracy with get_probs() on
        # the RAW model; the wrapper is used only to obtain gradients.
        got = _acc(net, X, y, BATCH)
        ref = MASTER[(MASTER.dataset == name) & (MASTER.model == mname) &
                     (MASTER.seed == seed) & (MASTER.attack == "clean")]
        ref = float(ref.accuracy.iloc[0])
        ok = abs(got - ref) < TOL
        VALID &= ok
        val_rows.append(dict(dataset=name, model=mname, seed=seed,
                             quantity="clean", reproduced=got, published=ref,
                             delta=got - ref, ok=ok))
        print(f"  {name:<14} {mname:<12} clean  got={got:.4f} "
              f"pub={ref:.4f}  d={got-ref:+.4f}  {'OK' if ok else 'FAIL'}")

        for kind in ("FGSM", "PGD"):
            eps = EPS_LIST[-1]
            got = attacked_acc(model, X, y, ATTACK_BS, kind, eps,
                               D["lo"], D["hi"], None,
                               tag=f"val|{name}|{mname}|{seed}|{kind}|{eps}")
            r = MASTER[(MASTER.dataset == name) & (MASTER.model == mname) &
                       (MASTER.seed == seed) & (MASTER.attack == kind) &
                       (np.isclose(MASTER.eps, eps))]
            ref = float(r.accuracy.iloc[0])
            # PGD has a random start, so its reproduction is stochastic
            # FGSM is deterministic and must agree.  PGD in the published run
            # began from an UNSEEDED uniform start, so its absolute value is
            # not reproducible after the fact; it is reported for information
            # and deliberately does NOT gate the grid.  E3 compares PGD arms
            # to each other under a shared start, which is what it needs.
            tol = TOL if kind == "FGSM" else 2e-2
            ok = abs(got - ref) < tol
            if kind == "FGSM":
                VALID &= ok
            val_rows.append(dict(dataset=name, model=mname, seed=seed,
                                 quantity=f"{kind}@{eps}", reproduced=got,
                                 published=ref, delta=got - ref, ok=ok))
            print(f"  {name:<14} {mname:<12} {kind}@{eps}  got={got:.4f} "
                  f"pub={ref:.4f}  d={got-ref:+.4f}  {'OK' if ok else 'FAIL'}")
        del model, net
        torch.cuda.empty_cache()

VAL = pd.DataFrame(val_rows)
print("\n" + ("VALIDITY CHECK PASSED (clean + FGSM reproduce the paper)"
               if VALID else
               "VALIDITY CHECK FAILED -- do not report the masked arm"))
if not VALID and MISSING_CK:
    print(f"\ncause: {len(MISSING_CK)} checkpoint(s) not found, not a numeric "
          f"mismatch. Check RESOLVED_VER (cell 4) or the path, then re-run.")
if not VALID:
    bad = VAL[(~VAL.ok) & (~VAL.quantity.str.startswith("PGD"))]
    print("\ngating cells that failed:")
    print(bad.round(4).to_string(index=False))
    if len(bad):
        print("\nlargest deviation:", f"{bad.delta.abs().max():.4f}",
              "| tolerance:", TOL)
    else:
        print("\nno gating cell deviated numerically; see the note above.")
display(VAL.round(4))


In [ ]:
# === 6. THE GRID: MASKED VERSUS UNMASKED =======================
# Runs end to end with no settings to change.  The work is ordered so the
# numbers that decide the paper's claim arrive first and the expensive tail is
# the least consequential part:
#
#   phase 1  FGSM on the two corpora that carry the argument      ~12 min
#   phase 2  PGD  on the same two                                 ~1.2 h
#   phase 3  FGSM on the negative-control corpus                  ~15 min
#   phase 4  PGD  on the negative-control corpus                  ~2.5 h
#
# Within a phase, models run cheapest first, so Mamba -- about 88% of the
# total cost, because its scan is a sequential Python loop -- is always last.
# After every phase the Transformer-Mamba margin is printed, so the answer is
# readable from roughly minute fifteen instead of only at the end.  Completed
# work is keyed per (dataset, model, seed, attack) and skipped, so a Colab
# disconnect costs at most the cell that was in flight.
assert VALID, ("The harness does not reproduce the published unmasked "
               "numbers; fix that before running the grid.")

CARRY   = [d for d in DATASETS if d != "CICIDS2018"]
CONTROL = [d for d in DATASETS if d == "CICIDS2018"]
PHASES  = ([(k, CARRY) for k in ("FGSM", "PGD") if CARRY] +
           [(k, CONTROL) for k in ("FGSM", "PGD") if CONTROL])
COST    = {"CNN1D": 0, "LSTM": 1, "Transformer": 2, "Mamba": 3}
ORDER   = sorted(MODELS, key=lambda m: COST.get(m, 9))

CSV = os.path.join(BASE, "results", f"e3_padding_mask_{RUN_ID}.csv")
rows, done = [], set()
if os.path.exists(CSV):
    prev = pd.read_csv(CSV)
    rows = prev.to_dict("records")
    for key, grp in prev.groupby(["dataset", "model", "seed", "attack"]):
        need = 1 if key[3] == "clean" else len(EPS_LIST) * 2
        if len(grp) >= need:
            done.add((key[0], key[1], int(key[2]), key[3]))
    print(f"resume: {len(rows)} rows on disk, {len(done)} completed units\n")
else:
    print("starting fresh\n")

def _margin(rs):
    """Transformer minus Mamba, over whatever attacked rows exist so far."""
    d = pd.DataFrame(rs)
    if d.empty or "attack" not in d:
        return None
    d = d[d.attack != "clean"]
    if d.empty:
        return None
    p = d.pivot_table(index="model", columns="masked", values="accuracy")
    if not {True, False} <= set(p.columns) or not {"Transformer", "Mamba"} <= set(p.index):
        return None
    return (p.loc["Transformer", False] - p.loc["Mamba", False],
            p.loc["Transformer", True] - p.loc["Mamba", True])

total = sum(len(ds) for _, ds in PHASES) * len(ORDER) * len(SEEDS) * len(EPS_LIST) * 2
t0, n_done, n_new = time.time(), 0, 0

for ph, (kind, dsets) in enumerate(PHASES, 1):
    print("=" * 72)
    print(f"PHASE {ph}/{len(PHASES)}: {kind} on {dsets}")
    print("=" * 72, flush=True)
    for name in dsets:
        D = DATA[name]
        X, y = D["X"].to(DEVICE), D["y"].long().to(DEVICE)
        m_on = D["mask"].to(DEVICE)
        for mname in ORDER:
            for seed in SEEDS:
                need_atk = (name, mname, seed, kind) not in done
                need_cln = (name, mname, seed, "clean") not in done
                if not need_atk and not need_cln:
                    n_done += len(EPS_LIST) * 2
                    continue
                ck = ckpt_for(name, mname, seed)
                if not os.path.exists(ck):
                    print("  !! missing", os.path.basename(ck))
                    n_done += len(EPS_LIST) * 2
                    continue
                net = build_model(mname)
                obj = torch.load(ck, map_location="cpu", weights_only=False)
                sd = obj
                if isinstance(obj, dict):
                    for k in ("model_state", "state_dict", "model_state_dict"):
                        if k in obj:
                            sd = obj[k]
                            break
                net.load_state_dict(sd, strict=True)
                net = net.to(DEVICE).eval()
                model = ModelWrapper(net, D["lo"], D["hi"]).eval()

                if need_cln:
                    rows.append(dict(dataset=name, model=mname, seed=seed,
                                     attack="clean", eps=0.0, masked=False,
                                     accuracy=_acc(net, X, y, BATCH)))
                if need_atk:
                    for eps in EPS_LIST:
                        for masked in (False, True):
                            # one tag per cell: both arms share one random start
                            a = attacked_acc(model, X, y, ATTACK_BS, kind, eps,
                                             D["lo"], D["hi"],
                                             m_on if masked else None,
                                             tag=f"{name}|{mname}|{seed}|{kind}|{eps}")
                            rows.append(dict(dataset=name, model=mname,
                                             seed=seed, attack=kind, eps=eps,
                                             masked=masked, accuracy=a))
                            n_done += 1
                            n_new += 1
                        um, mk = rows[-2]["accuracy"], rows[-1]["accuracy"]
                        el = time.time() - t0
                        rate = n_new / el if el > 0 else 0
                        eta = (total - n_done) / rate / 60 if rate > 0 else float("nan")
                        print(f"[{n_done:4d}/{total}] {name:<14}{mname:<12}"
                              f"s{seed} {kind:<5}eps={eps:<5} "
                              f"unmasked={um:.4f} masked={mk:.4f} "
                              f"delta={mk-um:+.4f} ({el/60:.1f}min, "
                              f"eta {eta:.0f}min)", flush=True)
                del model, net
                torch.cuda.empty_cache()
                pd.DataFrame(rows).to_csv(CSV, index=False)
    g = _margin(rows)
    if g:
        print(f"\n>>> after phase {ph}: Transformer - Mamba margin "
              f"unmasked {g[0]:+.4f} -> masked {g[1]:+.4f} "
              f"(change {g[1]-g[0]:+.4f})"
              f"{'   *** SIGN CHANGES ***' if g[0]*g[1] <= 0 else ''}\n",
              flush=True)

E3 = pd.DataFrame(rows)
E3.to_csv(CSV, index=False)
print(f"\nALL PHASES DONE in {(time.time()-t0)/60:.1f} min | rows: {len(E3)}")
print("saved ->", CSV)


In [ ]:
# === 7. AGGREGATE, AND THE EFFECT ON THE HEADLINE COMPARISON ====
atk = E3[E3.attack != "clean"]

_ds = sorted(atk.dataset.unique())
_at = sorted(atk.attack.unique())
_ep = sorted(atk.eps.unique())
_sd = sorted(atk.seed.unique())
print("=" * 72)
print("Mean white-box robust accuracy, masked vs unmasked")
print(f"  datasets: {_ds}")
print(f"  attacks : {_at}   budgets: {_ep}")
print(f"  seeds   : {len(_sd)}")
print("=" * 72)
piv = atk.pivot_table(index="model", columns="masked", values="accuracy")
piv.columns = ["unmasked", "masked"]
piv["delta"] = piv["masked"] - piv["unmasked"]
display(piv.sort_values("unmasked", ascending=False).round(4))

print("\nPer dataset and attack:")
d2 = atk.pivot_table(index=["dataset", "attack", "model"],
                     columns="masked", values="accuracy")
d2.columns = ["unmasked", "masked"]
d2["delta"] = d2["masked"] - d2["unmasked"]
display(d2.round(4))

print("\nPer budget (how the artefact scales with eps):")
d3 = atk.pivot_table(index=["model", "eps"], columns="masked",
                     values="accuracy")
d3.columns = ["unmasked", "masked"]
d3["delta"] = d3["masked"] - d3["unmasked"]
display(d3.round(4))

# The claim the paper rests on: Transformer minus Mamba.
print("=" * 72)
if {"Transformer", "Mamba"} <= set(piv.index):
    gu = piv.loc["Transformer", "unmasked"] - piv.loc["Mamba", "unmasked"]
    gm = piv.loc["Transformer", "masked"] - piv.loc["Mamba", "masked"]
    print(f"Transformer - Mamba margin: unmasked {gu:+.4f} -> masked {gm:+.4f}"
          f"  (change {gm-gu:+.4f})")
    if gm * gu <= 0:
        print("  *** THE SIGN OF THE MARGIN CHANGES. The headline claim must")
        print("      be revisited before submission. ***")
    elif abs(gm - gu) > 0.05:
        print("  *** The margin moves by more than 5 points; report the")
        print("      masked figures alongside the unmasked ones. ***")
    else:
        print("  The margin is stable under masking; the caveat can be")
        print("  reported as closed.")

# Does the ordering survive?
ou = list(piv.sort_values("unmasked", ascending=False).index)
om = list(piv.sort_values("masked", ascending=False).index)
print(f"\nordering unmasked: {ou}\nordering masked:   {om}")
print("ordering preserved:", ou == om)


In [ ]:
# === 8. EXPORT: JSON, CSV AND A LATEX TABLE =====================
# LaTeX is emitted only if the validity check passed, so a failed
# reproduction cannot silently reach the manuscript.
assert VALID, "validity check failed; refusing to emit LaTeX"

out_dir = os.path.join(BASE, "results")
stamp = time.strftime("%Y%m%d_%H%M%S")

payload = dict(
    run_id=RUN_ID, generated=stamp,
    config=dict(L=SEQ_LEN, d_model=D_MODEL, n_layers=N_LAYERS,
                eps_list=EPS_LIST, pgd_alpha=PGD_ALPHA, pgd_steps=PGD_STEPS,
                datasets=DATASETS, models=MODELS, seeds=SEEDS,
                batch=BATCH, device=torch.cuda.get_device_name(0)),
    padding={k: dict(d=v["d"], n_pad=v["n_pad"],
                     frac=round(v["n_pad"] / SEQ_LEN, 4), n_test=v["n"])
             for k, v in DATA.items()},
    validity=VAL.to_dict(orient="records"),
    aggregate=piv.round(6).to_dict(orient="index"),
    per_cell=d2.round(6).reset_index().to_dict(orient="records"),
)
jpath = os.path.join(out_dir, f"e3_padding_mask_{stamp}.json")
with open(jpath, "w") as f:
    json.dump(payload, f, indent=1, default=float)
print("JSON ->", jpath)

# --- LaTeX table body, ready to \input or paste -------------------
lines = []
for mdl in ["Mamba", "Transformer", "CNN1D", "LSTM"]:
    if mdl not in piv.index:
        continue
    disp = {"LSTM": "BiLSTM"}.get(mdl, mdl)
    r = piv.loc[mdl]
    lines.append(f"{disp} & {r['unmasked']:.4f} & {r['masked']:.4f} & "
                 f"{r['delta']:+.4f} \\\\")
body = "\n".join(lines).rstrip("\\\\") + "%"
tpath = os.path.join(out_dir, f"e3_padding_mask_{stamp}.tex")
with open(tpath, "w") as f:
    f.write(body + "\n")
print("LaTeX ->", tpath)
print()
print(body)

print("""
----------------------------------------------------------------------
Send back to the review thread:
  * this cell's printed output,
  * e3_padding_mask_<stamp>.json,
  * the validity table from cell 5.
The manuscript edits that follow from the result -- Section 3.2.3, the
limitations table, and if the margin moved, Sections 4.3 and 5.2 -- are
prepared once these are in hand.
----------------------------------------------------------------------""")
